In [1]:
from pathlib import Path
import importlib
import src.flow.builder as fb

importlib.reload(fb)
from src.flow.builder import FlowBuilder

import json
import hashlib
import pandas as pd
import numpy as np
import yaml

from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.datasets.pcap_reader import iter_packets

paths = load_paths()
paths.ensure_dirs()
logger = setup_logger(level="INFO")

logger.info(f"Repo root: {paths.repo_root}")
logger.info(f"ISCX raw dir: {paths.data_raw_iscx}")
logger.info(f"Processed dir: {paths.data_processed}")

features_path = paths.configs_dir / "features.yaml"
assert features_path.exists(), f"Missing: {features_path}"

features_cfg = yaml.safe_load(features_path.read_text()) or {}
w = features_cfg.get("window") or {}

N = int(w.get("N", 100))
EPS = float(w.get("eps", 1e-6))
MIN_PACKETS = int(w.get("min_packets", 10))

logger.info(f"Loaded features config: N={N}, eps={EPS}, min_packets={MIN_PACKETS}")

raw_iscx = paths.data_raw_iscx
vpn_dir = raw_iscx / "vpn"
nonvpn_dir = raw_iscx / "nonvpn"

assert vpn_dir.exists(), f"Missing folder: {vpn_dir}"
assert nonvpn_dir.exists(), f"Missing folder: {nonvpn_dir}"


def list_pcaps(d: Path):
    exts = {".pcap", ".pcapng"}
    return sorted([p for p in d.rglob("*") if p.suffix.lower() in exts])


vpn_pcaps = list_pcaps(vpn_dir)
nonvpn_pcaps = list_pcaps(nonvpn_dir)

logger.info(f"VPN pcaps: {len(vpn_pcaps)}")
logger.info(f"NonVPN pcaps: {len(nonvpn_pcaps)}")

print(vpn_pcaps[:5])
print(nonvpn_pcaps[:5])


def make_non_decreasing(ts, eps: float):
    out = []
    prev = None
    for t in ts:
        t = float(t)
        if prev is None:
            out.append(t)
            prev = t
            continue
        if t < prev:
            t = prev + eps
        out.append(t)
        prev = t
    return out


def normalize_capture_name(name: str) -> str:
    s = str(name).strip().lower().replace("\\", "/").split("/")[-1]
    return s


def derive_app_from_prefixed_filename(fname: str) -> str:
    s = normalize_capture_name(fname)
    s = s.replace(".pcapng", "").replace(".pcap", "")
    s = s.replace("vpn_", "").replace("nonvpn_", "")
    return s.split("_", 1)[0]


def conn_to_str(conn) -> str:
    try:
        ip_a, port_a, ip_b, port_b, proto = conn
        return f"{ip_a}:{int(port_a)}-{ip_b}:{int(port_b)}-p{int(proto)}"
    except Exception:
        return str(conn)


def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


test_pcap = nonvpn_pcaps[0]
logger.info(f"Testing one PCAP: {test_pcap}")

builder = FlowBuilder(tcp_timeout=1800.0, udp_timeout=60.0)

n_pkts = 0
for rec in iter_packets(test_pcap):
    builder.add_packet(**rec)
    n_pkts += 1

flows_list = builder.finalize()
logger.info(f"Packets read: {n_pkts}")
logger.info(f"Flows built: {len(flows_list)}")

print(flows_list[0].keys())
print(flows_list[0]["connection"])

rows = []


def process_pcap(pcap_path: Path, label: int):
    pref = "vpn_" if label == 1 else "nonvpn_"
    base = pcap_path.name
    file_name = pref + base

    builder = FlowBuilder(tcp_timeout=1800.0, udp_timeout=60.0)

    pkt_count = 0
    for rec in iter_packets(pcap_path):
        builder.add_packet(**rec)
        pkt_count += 1

    built = builder.finalize()

    for f in built:
        ts = make_non_decreasing(f["timestamps"], eps=EPS)
        rows.append({
            "connection": f["connection"],
            "timestamps": ts,
            "sizes": f["sizes"],
            "directions": f["directions"],
            "file_names": file_name,
            "label": int(label),
        })

    return pkt_count, len(built)


total_pkts = 0
total_flows = 0

for i, p in enumerate(nonvpn_pcaps):
    pk, fl = process_pcap(p, label=0)
    total_pkts += pk
    total_flows += fl
    if (i + 1) % 10 == 0:
        logger.info(f"NonVPN processed: {i + 1}/{len(nonvpn_pcaps)} | pkts={total_pkts} flows={total_flows}")

for i, p in enumerate(vpn_pcaps):
    pk, fl = process_pcap(p, label=1)
    total_pkts += pk
    total_flows += fl
    if (i + 1) % 10 == 0:
        logger.info(f"VPN processed: {i + 1}/{len(vpn_pcaps)} | pkts={total_pkts} flows={total_flows}")

logger.info(f"TOTAL pkts={total_pkts} TOTAL flows={total_flows}")

df = pd.DataFrame(rows)
print(df.head())
print(df.shape)

df = df.reset_index(drop=True)
df["row_id"] = df.index.astype("int64")

expected_cols = {"connection", "timestamps", "sizes", "directions", "file_names", "label", "row_id"}
missing = expected_cols - set(df.columns)
assert not missing, f"Missing columns: {missing}"


def is_listlike(x):
    return isinstance(x, (list, tuple))


for col in ["timestamps", "sizes", "directions"]:
    bad = df[~df[col].map(is_listlike)]
    assert len(bad) == 0, f"Non-list entries in {col}: {bad.head()}"

lens = pd.DataFrame({
    "t": df["timestamps"].map(len),
    "s": df["sizes"].map(len),
    "d": df["directions"].map(len),
})
mismatch = df[(lens["t"] != lens["s"]) | (lens["t"] != lens["d"])]
assert len(mismatch) == 0, f"Mismatched list lengths:\n{mismatch.head()}"

bad_dir = df[~df["directions"].map(lambda dirs: set(dirs).issubset({0, 1}))]
assert len(bad_dir) == 0, f"Invalid direction values:\n{bad_dir.head()}"


def sizes_valid(sz) -> bool:
    if any(x is None for x in sz):
        return False
    return all((isinstance(x, (int, float)) and x >= 0 and x < 65536) for x in sz)


bad_sizes = df[~df["sizes"].map(sizes_valid)]
assert len(bad_sizes) == 0, f"Invalid sizes:\n{bad_sizes[['file_names', 'sizes']].head()}"

df["file_names"] = df["file_names"].astype(str)
df["capture_name"] = df["file_names"].map(normalize_capture_name)
df["capture_id"] = df["capture_name"]
df["connection_str"] = df["connection"].map(conn_to_str)
df["flow_key"] = df["connection_str"]
df["flow_id"] = df["capture_id"] + "::" + df["row_id"].astype(str)
df["app"] = df["file_names"].map(derive_app_from_prefixed_filename)
df["packet_count_full"] = df["sizes"].map(len)

# windowing
df["timestamps"] = df["timestamps"].map(lambda xs: xs[:N])
df["sizes"] = df["sizes"].map(lambda xs: xs[:N])
df["directions"] = df["directions"].map(lambda xs: xs[:N])

df["packet_count"] = df["sizes"].map(len)
df["min_packets_ok"] = df["packet_count"] >= MIN_PACKETS

logger.info(f"ISCX flows built: shape={df.shape}")
logger.info("Label counts:\n" + str(df["label"].value_counts()))
logger.info(f"min_packets_ok rate: {100 * df['min_packets_ok'].mean():.2f}%")

print("\n--- SANITY CHECK: Label vs Filename ---")
df["is_vpn_file"] = df["file_names"].str.startswith("vpn_")
crosstab = pd.crosstab(df["is_vpn_file"], df["label"])
print(crosstab)

if crosstab.shape == (2, 2):
    nonvpn_ok = crosstab.loc[False, 0] > 0 and crosstab.loc[False, 1] == 0
    vpn_ok = crosstab.loc[True, 1] > 0 and crosstab.loc[True, 0] == 0

    if nonvpn_ok and vpn_ok:
        print("\nSUCCESS: Labels match filename prefixes perfectly.")
    else:
        print("\nWARNING: Mismatch detected between filename prefix and label!")
else:
    print("\nWARNING: Crosstab shape is unexpected. Check manually.")

print("\n--- App Distribution by Label ---")
print(pd.crosstab(df["app"], df["label"]))

flows = df[
    [
        "capture_id",
        "capture_name",
        "row_id",
        "flow_id",
        "flow_key",
        "connection_str",
        "timestamps",
        "sizes",
        "directions",
        "file_names",
        "app",
        "label",
        "packet_count",
        "packet_count_full",
        "min_packets_ok",
    ]
].copy()

assert flows["flow_id"].is_unique

flows["timestamps"] = flows["timestamps"].map(lambda xs: [float(x) for x in xs])
flows["sizes"] = flows["sizes"].map(lambda xs: [int(x) for x in xs])
flows["directions"] = flows["directions"].map(lambda xs: [int(x) for x in xs])

# UDP noise filtering requires parsing connection tuple before dropping raw connection
df_conn = df[["flow_id", "connection"]].copy()
df_conn["proto"] = df_conn["connection"].map(lambda c: int(c[4]))
df_conn["port_a"] = df_conn["connection"].map(lambda c: int(c[1]))
df_conn["port_b"] = df_conn["connection"].map(lambda c: int(c[3]))
df_conn = df_conn.drop(columns=["connection"])

flows = flows.merge(df_conn, on="flow_id", how="left", validate="one_to_one")

noise_ports = {5355, 5353, 137, 138, 1900}
is_udp = flows["proto"] == 17
has_noise_port = flows["port_a"].isin(noise_ports) | flows["port_b"].isin(noise_ports)

flows_filtered = flows[~(is_udp & has_noise_port)].copy()

# exact duplicate removal
flows_filtered["sizes_tuple"] = flows_filtered["sizes"].apply(tuple)
flows_filtered["dirs_tuple"] = flows_filtered["directions"].apply(tuple)
flows_filtered["ts_tuple"] = flows_filtered["timestamps"].apply(lambda x: tuple(np.round(x, 6)))

before_dedup = len(flows_filtered)
flows_filtered = flows_filtered.drop_duplicates(
    subset=["capture_id", "connection_str", "sizes_tuple", "dirs_tuple", "ts_tuple"]
).copy()
after_dedup = len(flows_filtered)

flows_filtered = flows_filtered.drop(columns=["sizes_tuple", "dirs_tuple", "ts_tuple"])

logger.info(f"Duplicate removal: {before_dedup} -> {after_dedup} flows (dropped {before_dedup - after_dedup})")

print("\nBefore:", len(flows), "flows | min_packets_ok:", flows["min_packets_ok"].mean())
print("After :", len(flows_filtered), "flows | min_packets_ok:", flows_filtered["min_packets_ok"].mean())

print("\npacket_count top 20 (filtered):")
print(flows_filtered["packet_count"].value_counts().head(20))

print("\nmin_packets_ok by label (filtered):")
print(flows_filtered.groupby("label")["min_packets_ok"].mean())

# drop diagnostics not needed downstream
flows_filtered = flows_filtered.drop(columns=["proto", "port_a", "port_b"], errors="ignore")

# force canonical dtypes
flows_filtered["flow_id"] = flows_filtered["flow_id"].astype(str)
flows_filtered["capture_id"] = flows_filtered["capture_id"].astype(str)
flows_filtered["capture_name"] = flows_filtered["capture_name"].astype(str)
flows_filtered["flow_key"] = flows_filtered["flow_key"].astype(str)
flows_filtered["connection_str"] = flows_filtered["connection_str"].astype(str)
flows_filtered["file_names"] = flows_filtered["file_names"].astype(str)
flows_filtered["app"] = flows_filtered["app"].astype(str)
flows_filtered["label"] = flows_filtered["label"].astype(int)
flows_filtered["packet_count"] = flows_filtered["packet_count"].astype(int)
flows_filtered["packet_count_full"] = flows_filtered["packet_count_full"].astype(int)
flows_filtered["min_packets_ok"] = flows_filtered["min_packets_ok"].astype(bool)

out_dir = paths.data_processed / "iscx"
out_dir.mkdir(parents=True, exist_ok=True)

flows_path = out_dir / "flows.parquet"
flows_filtered.to_parquet(flows_path, index=False)
logger.info(f"Saved ISCX flows parquet: {flows_path}")

manifest = {
    "dataset": "iscx",
    "flows_parquet": str(flows_path),
    "flows_sha256": sha256_file(flows_path),
    "features_yaml": str(features_path),
    "features_yaml_sha256": hashlib.sha256(features_path.read_bytes()).hexdigest(),
    "rows": int(len(flows_filtered)),
    "unique_captures": int(flows_filtered["capture_id"].nunique()),
    "unique_flows": int(flows_filtered["flow_id"].nunique()),
    "label_counts": flows_filtered["label"].value_counts().to_dict(),
    "window": {"N": int(N), "eps": float(EPS), "min_packets": int(MIN_PACKETS)},
    "pct_min_packets_ok": float(flows_filtered["min_packets_ok"].mean() * 100),
}

manifest_path = out_dir / "flows_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
logger.info(f"Saved ISCX flows manifest: {manifest_path}")

print(flows_filtered.head())

from src.splits.make_split_iscx import make_iscx_capture_split, write_capture_lists

logger.info("Creating canonical ISCX capture splits...")
splits = make_iscx_capture_split(flows_path, seed=42)
write_capture_lists(splits, paths.data_splits, prefix="iscx")

print("ISCX split counts:", {k: len(v) for k, v in splits.items()})
print("Created split files:")
print(paths.data_splits / "iscx_train_captures.txt")
print(paths.data_splits / "iscx_val_captures.txt")
print(paths.data_splits / "iscx_test_captures.txt")

2026-03-29 20:40:16 | INFO | ai-vpn-firewall | Repo root: C:\Users\scoti\PycharmProjects\ai-vpn-firewall
2026-03-29 20:40:16 | INFO | ai-vpn-firewall | ISCX raw dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\raw\iscx
2026-03-29 20:40:16 | INFO | ai-vpn-firewall | Processed dir: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\data\processed
2026-03-29 20:40:16 | INFO | ai-vpn-firewall | Loaded features config: N=100, eps=1e-06, min_packets=3
2026-03-29 20:40:16 | INFO | ai-vpn-firewall | VPN pcaps: 31
2026-03-29 20:40:16 | INFO | ai-vpn-firewall | NonVPN pcaps: 109
[WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_aim_chat1a.pcap'), WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_aim_chat1b.pcap'), WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_bittorrent.pcap'), WindowsPath('C:/Users/scoti/PycharmProjects/ai-vpn-firewall/data/raw/iscx/vpn/vpn_email2a.pcap'), WindowsPath('C:/Us